# 12 — Полный цикл обучения CrafText: rollout → replay → token PPO update

Эта тетрадка связывает текущий synchronous CrafText pipeline в один обучающий цикл: batched CrafText rollout, сохранение replay evidence, преобразование replay в `TextTrajectoryBatch`, расчёт returns и один masked token PPO update. Она намеренно маленькая: цель — доказать контракты данных и маски, а не обучить production Qwen/RLCluster workload.

> Для реального Qwen backend задайте локальный snapshot. Ячейка ниже также содержит deterministic toy backend, чтобы цикл можно было прочитать и адаптировать без приватных весов.

In [ ]:
from pathlib import Path
import jax.numpy as jnp
from tunix_craftext.algorithms import masked_token_ppo_loss, masked_token_returns
from tunix_craftext.batched_rollout import collect_batched_text_rollout, replays_from_batched_rollout
from tunix_craftext.config import load_mvp_config
from tunix_craftext.llm import LlmResponse
from tunix_craftext.prompts import MegaPromptRenderer
from tunix_craftext.replay import save_replay
from tunix_craftext.runtime import build_craftext_runtime
from tunix_craftext.text_trajectory import text_trajectory_from_replay

In [ ]:
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook inside tunix-craftext.')
config = load_mvp_config(ROOT / 'configs/mvp/qwen_craftext.yaml')
runtime = build_craftext_runtime(config)
print('CrafText actions:', runtime.actions.labels)

In [ ]:
class ScriptedTrainingBackend:
    '''Deterministic stand-in for a batched Qwen sampler with token provenance.'''
    def __init__(self):
        self.calls = 0
    def complete(self, request):
        return self.complete_batch((request,))[0]
    def complete_batch(self, requests):
        self.calls += 1
        responses = []
        for row, _ in enumerate(requests):
            # Alternate valid CrafText labels while retaining token/logprob evidence.
            label = runtime.actions.labels[(self.calls + row) % len(runtime.actions.labels)]
            responses.append(LlmResponse(
                raw_text=f'<action>{label}</action>',
                model='scripted-training-backend',
                backend='notebook',
                token_ids=(100 + self.calls, 200 + row),
                token_logprobs=(-0.4, -0.2),
                prompt_token_ids=(1, 2, 3, 4),
            ))
        return tuple(responses)

backend = ScriptedTrainingBackend()

In [ ]:
rollout = collect_batched_text_rollout(
    runtime.adapter,
    MegaPromptRenderer(config.prompt.template),
    backend,
    actions=runtime.actions,
    batch_size=2,
    horizon=3,
    seed=config.run.seed,
    goal='Collect useful CrafText experience for a tiny PPO update.',
    max_new_tokens=8,
    invalid_action='fallback',
    fallback_action_id=runtime.actions.index_of('NOOP'),
)
print('LLM batch calls:', backend.calls)
for t, decision in enumerate(rollout.decisions):
    print('t=', t, 'actions=', [action.label for action in decision.actions], 'reward=', decision.transition.reward.tolist())

In [ ]:
output_dir = ROOT / 'artifacts/trajectories/full-cycle-craftext-training'
replays = replays_from_batched_rollout(rollout, config_path='configs/mvp/qwen_craftext.yaml', commit='notebook', backend='scripted')
for env_index, replay in enumerate(replays):
    save_replay(output_dir / f'env-{env_index}.json', replay)
    print('saved replay', env_index, 'steps=', len(replay.steps))

In [ ]:
batches = tuple(text_trajectory_from_replay(replay) for replay in replays)
batch = batches[0]
returns = masked_token_returns(batch.rewards, batch.token_mask, gamma=0.99)
advantages = returns  # Smoke update: replace with critic baseline once value bridge is trainable.
zeros = jnp.zeros_like(batch.old_logprobs)
loss, metrics = masked_token_ppo_loss(
    batch.old_logprobs,
    batch.old_logprobs,
    advantages,
    zeros,
    zeros,
    returns,
    batch.policy_mask,
    clip_epsilon=0.2,
    value_coefficient=0.5,
    entropy=zeros,
    entropy_coefficient=0.01,
)
print('token_ids:', batch.token_ids.tolist())
print('policy_mask:', batch.policy_mask.tolist())
print('returns:', returns.tolist())
print('loss:', float(loss))
print({name: float(value) for name, value in metrics.items()})

## Что заменяется в production loop

- `ScriptedTrainingBackend` заменяется на `QwenTunixBackend` или future `RLCluster.ROLLOUT` actor.
- `advantages = returns` заменяется на trainable critic/value bridge.
- Notebook сохраняет replay evidence до loss, чтобы audit мог проверить prompt, action, token ids/logprobs, rewards и fallback masks.